In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('glass.csv')

In [3]:
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [4]:
df.shape

(214, 10)

In [ ]:
# we are converting multiple glass types into a binary problem
df["y"] = (df["Type"] ==1).astype(int)
df = df.drop(columns=["Type"])

In [ ]:
# separating input and output values
X = df.drop(columns=["y"]).values
y = df["y"].values

In [ ]:
# testing happens on unseen data as testing on training data gives false confidence
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [ ]:
# glass features have different numeric ranges so if we dont scale learning becomes unstable
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# sigmoid converts score into value between 0 and 1 and is used mostly in binary class classification
import numpy as np
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# forward computation
def predict_proba(X, w, b):
    z = X @ w + b # @ equivalent to np.dot(X,w)
    p = sigmoid(z)
    return p

In [ ]:
# we no longer ask is our prediction right or wrong?
# loss function tells how wrong is our confidence 
def loss(y, p):
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p)) # binary cross entropy

In [13]:
# we are updating weights to propagate error in back direction
def update_weights(X, y, w, b, lr):
# compute predictions
    p = predict_proba(X, w, b)
# compute error
    error = p - y # tells direction and strength of correction
# update weights and bias
# learning rate controls step size
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    return w, b


In [17]:
# this loop performs gradient descent optimisation so weights slightly change towards better values
# this loop repeatedly updates model parameters using gradient descent so that prediction error decreases and the model learns optimal weights
# initialize weights and bias
w = np.zeros(X_train.shape[1])
b =0.0
lr =0.1
epochs =100
for _ in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)

In [ ]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)
# The model outputs probabilities, not decisions
# A threshold converts probability into a class label
# In glass quality control, a higher threshold is safer because only highly confident predictions are accepted as good products, reducing the risk of defective glass passing inspection.

In [ ]:
# Unlike a perceptron, which produces only hard binary outputs (0 or 1) using a step activation function,
# logistic regression uses a sigmoid activation that outputs probabilities between 0 and 1, allowing smoother learning and 
# confidence-based predictions. 
# The sigmoid function matters because it makes the model differentiable, enabling gradient descent optimization and 
# probabilistic interpretation of predictions rather than rigid decisions.
# However, one major problem still remains unsolved: both logistic regression and perceptrons can only learn linearly separable 
# relationships, meaning they cannot model complex nonlinear patterns (such as XOR) without adding feature transformations 
# or deeper neural network layers.